# Task 3 — CycleGAN image style transfer (Monet ↔ Photo), built from scratch
**DATA266 Lab 1 · Member: Shibin Thomas**

The logic lives in tested modules next to this notebook (`data.py`, `models.py`, `train.py`, `translate.py`,
`gan_metrics.py`, `human_audit.py`) and in `../evaluate_local.py`; the notebook calls them, so it gives the same
results as the one-command CLI `python task3_gan/shibin_thomas/src/run_pipeline.py`.

| Brief | Where |
|---|---|
| 3.1.1 Two unpaired domains | §1 |
| 3.1.2 CycleGAN: 2 generators + 2 discriminators | §2, `models.py` |
| 3.1.3 Adversarial + cycle-consistency (+ identity) losses | §3, `train.py` |
| 3.1.4 Translation between the domains | §4, `pred_A2B/`, `pred_B2A/` |
| 3.1.5 / 3.2.3 Training behaviour, convergence, stability | §3 |
| 3.2.1 Visual quality · 3.2.2 cycle-consistency verification | §4–5 |
| 3.2.5 Kaggle submission · 3.2.6 blinded human audit | §6–7 |
| All Task 3 metrics (both directions) | §5, `full_metrics_report.csv` |

Domain A = **Monet**, domain B = **Photo** (same convention as the instructor's evaluation script).
**No pretrained network generates or touches the images** — the generators and discriminators are trained
from random initialisation; pretrained Inception / LPIPS are used only to *measure* the outputs.

In [ ]:
CONFIG = "auto"   # "auto" = config of the run whose images are in outputs/pred_* (the submitted run); or a path
OVERRIDES = []
RUN_TRAINING = "auto"    # "auto": train only if no evaluated run of this config exists; True / False to force
RUN_ID = "latest"

In [ ]:
import sys, json
from pathlib import Path
here = Path.cwd().resolve()
REPO = next((p for p in [here, *here.parents] if (p / "task3_gan").is_dir()), None)
assert REPO is not None, "open this notebook from inside the cloned repository"
MEMBER = REPO / "task3_gan" / "shibin_thomas"
sys.path.insert(0, str(MEMBER / "src")); sys.path.insert(0, str(MEMBER))

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from PIL import Image as PILImage

from utils import load_config, repo_path, read_json, run_dirs, get_device, hardware_info
from data import list_images
from models import build_models, count_params
if CONFIG == "auto":
    info_p = MEMBER / "outputs" / "predictions_info.json"
    sub_run = json.loads(info_p.read_text(encoding="utf-8"))["run_id"] if info_p.exists() else None
    CONFIG = "task3_gan/shibin_thomas/configs/cyclegan_v1.yaml"
    if sub_run and (REPO / f"task3_gan/shibin_thomas/configs/{sub_run.rsplit('_', 1)[0]}.yaml").exists():
        CONFIG = f"task3_gan/shibin_thomas/configs/{sub_run.rsplit('_', 1)[0]}.yaml"
        if RUN_ID == "latest":
            RUN_ID = sub_run
print("config:", CONFIG, "| run:", RUN_ID)
cfg = load_config(CONFIG, OVERRIDES)
device = get_device("auto")
print("device:", device, "|", hardware_info(device).get("gpu_name") or hardware_info(device)["cpu_model"], "| torch", torch.__version__)

## 1. Two unpaired image domains (3.1.1)
A = 300 Monet paintings, B = 7,038 photos, all 256×256 RGB. There is **no pairing** between the domains: every
training step draws an independent random Monet and a random photo. Augmentation: resize to 286, random 256 crop,
random horizontal flip; pixels scaled to [−1, 1].

In [ ]:
pa, pb = list_images(repo_path(cfg["data"]["monet_dir"])), list_images(repo_path(cfg["data"]["photo_dir"]))
print(f"domain A (Monet): {len(pa):,} images | domain B (Photo): {len(pb):,} images")
sizes = {PILImage.open(p).size for p in pa[:50] + pb[:50]}
print("image sizes (first 50 of each):", sizes)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for j in range(6):
    axes[0, j].imshow(PILImage.open(pa[j])); axes[0, j].axis("off")
    axes[1, j].imshow(PILImage.open(pb[j])); axes[1, j].axis("off")
axes[0, 0].set_title("A: Monet", loc="left"); axes[1, 0].set_title("B: Photo", loc="left")
fig.tight_layout(); display(fig); plt.close(fig)

## 2. CycleGAN architecture (3.1.2)
* **Generators** `G_AB` (Monet→Photo) and `G_BA` (Photo→Monet): ResNet encoder–decoder — c7s1-64, two stride-2
  downsampling convs, **9 residual blocks** at 64×64, two upsampling stages, c7s1-3 + tanh; InstanceNorm, reflection
  padding. Upsampling = **nearest-neighbour ×2 + 3×3 conv (resize-convolution)** instead of transposed convolutions,
  to avoid checkerboard artifacts.
* **Discriminators** `D_A` (Monet) and `D_B` (Photo): **70×70 PatchGAN** — scores every overlapping 70×70 patch as
  real/fake (30×30 output map for a 256×256 image), which focuses the adversarial signal on texture/brush strokes.

In [ ]:
nets = build_models(cfg["model"])
for k, v in nets.items():
    print(f"{k}: {count_params(v):,} parameters")
with torch.no_grad():
    print("G output shape:", tuple(nets["G_AB"](torch.randn(1, 3, 256, 256)).shape),
          "| D output (patch grid):", tuple(nets["D_A"](torch.randn(1, 3, 256, 256)).shape))
print(nets["G_AB"])

## 3. Training — adversarial + cycle-consistency + identity losses (3.1.3, 3.1.5)
`L_G = LSGAN(D_B(G_AB(a))) + LSGAN(D_A(G_BA(b))) + λ_cyc·(|G_BA(G_AB(a)) − a| + |G_AB(G_BA(b)) − b|) + λ_id·(|G_BA(a) − a| + |G_AB(b) − b|)`;
discriminators use least-squares loss against a 50-image history pool. Adam (2e-4, β1 = 0.5), constant LR, then
linear decay to 0; 2,000 unpaired pairs per epoch, batch 4. The exact λ values, epochs and the v2 additions
(DiffAugment on the discriminators, generator EMA, held-out epoch selection) are printed from the config below;
`results.md` §3 and §5b justify them.
Raw logs: `reproducibility/raw_logs/task3_gan/shibin_thomas/<run_id>/`.

In [ ]:
from utils import MEMBER_DIR
finished = sorted(p.name for p in (MEMBER_DIR / "outputs").glob(cfg["run_name"] + "_*")
                  if (p / "eval_metrics.json").exists())
print({k: cfg["training"].get(k) for k in ["epochs_constant", "epochs_decay", "lambda_cycle", "lambda_identity",
                                           "diffaug_A", "diffaug_B", "ema_decay"]}, "| selection:", cfg.get("selection"))
if RUN_TRAINING == "auto":
    RUN_TRAINING = not finished
if RUN_TRAINING:
    from run_pipeline import run_pipeline
    RUN_ID = run_pipeline(CONFIG, OVERRIDES)
elif RUN_ID == "latest":
    RUN_ID = finished[-1]
dirs = run_dirs(RUN_ID)
print("trained now" if RUN_TRAINING else "re-using finished run", "->", RUN_ID)
ep = pd.read_csv(dirs["raw_logs"] / "epochs.csv")
ep

In [ ]:
display(Image(filename=str(dirs["outputs"] / "loss_curves.png")))
display(Image(filename=str(dirs["outputs"] / "epoch_curves.png")))
summ = read_json(dirs["outputs"] / "train_summary.json")
print({k: summ[k] for k in ["epochs", "total_steps", "nonfinite_steps", "training_time_sec", "train_images_per_sec"]})
print("GPU at start:", summ.get("gpu_at_start"))

In [ ]:
# how the translations evolve during training (rows: photo, photo->Monet, Monet, Monet->photo)
samples = sorted((dirs["outputs"] / "samples").glob("epoch_*.png"))
for p in [samples[0], samples[len(samples) // 2], samples[-1]]:
    print(p.name); display(Image(filename=str(p)))

## 4. Translation and visual quality (3.1.4, 3.2.1)
`translate.py` writes `outputs/pred_A2B/` (G_AB applied to the 300 Monets) and `outputs/pred_B2A/` (G_BA applied to the
first 300 photos) — exactly the images the instructor's script scores. Rows below: input, translation, reconstruction.

In [ ]:
for tag, title in (("B2A", "Photo -> Monet"), ("A2B", "Monet -> Photo")):
    print(title, "(rows: input / translation / cycle reconstruction)")
    display(Image(filename=str(dirs["outputs"] / f"final_samples_{tag}.png")))
print(read_json(repo_path(cfg["export"]["out_dir"]) / "predictions_info.json"))

## 5. Evaluation — all metrics, both directions (3.2.2–3.2.4)
FID / "MiFID" follow the instructor's script exactly; KID, precision/recall, density/coverage, LPIPS, cycle L1 and
content cosine similarity come from `gan_metrics.py`. **Cycle-consistency verification:** the cycle L1 and LPIPS between
an input and its reconstruction should be small, and the reconstructions above should match the inputs.

In [ ]:
ev = read_json(dirs["outputs"] / "eval_metrics.json")
print("Kaggle submission:", ev["submission"])
display(pd.DataFrame(ev["directions"]))
display(pd.DataFrame(ev["training"], index=["value"]).T)

In [ ]:
fc = (dirs["outputs"] / "failure_candidates.md").read_text(encoding="utf-8")
display(Markdown(fc.replace("](failure_", f"]({(dirs['outputs']).as_posix()}/failure_")[:3000]))
for f in sorted(dirs["outputs"].glob("failure_*.png"))[:3]:
    print(f.name); display(Image(filename=str(f)))

## 6. Blinded human audit (3.2.6)
`human_audit.py make` builds 30 fixed, shuffled, anonymised panels (S01–S30, input | output) with a hidden key. Two
raters score style, content and artifacts (1–5) independently in `rater1.csv` / `rater2.csv`; `human_audit.py score`
computes the mean scores and inter-rater agreement (quadratic-weighted Cohen's κ, exact and ±1 agreement).

In [ ]:
audit_dir = repo_path(cfg["export"]["out_dir"]) / "human_audit"
res = audit_dir / "human_audit_results.json"
if res.exists():
    display(pd.DataFrame(read_json(res)["criteria"]).T)
else:
    print("Human audit pending -> see", (audit_dir / "INSTRUCTIONS.md").relative_to(REPO))
display(Image(filename=str(audit_dir / "contact_sheet.jpg"), width=900))

## 7. Kaggle submission (3.2.5)
`submission.csv` (ID, FID, MiFID) is produced by `evaluate_local.py` with the instructor's protocol from the direct
outputs of my generators. After uploading it, the leaderboard score and rank are recorded in `kaggle_leaderboard.json`.

In [ ]:
sub = MEMBER / "submission.csv"
if sub.exists() and not cfg.get("smoke"):
    display(pd.read_csv(sub))
print(json.dumps(read_json(MEMBER / "kaggle_leaderboard.json"), indent=1))